# Switch temperatures with Temperature-Dependent Power Flow (TDPF)

The TDPF of pandapower solves the conductor temperature of the overhead lines, because the line
resistance depends on it. Switches are treated differently: their resistance is not corrected with
the temperature, but the switch temperature is an interesting result on its own, e.g. for a loading
assessment according to IEC 62271-306.

When `runpp` is called with `tdpf=True`, pandapower reports the temperature of every current carrying
switch in `net.res_switch.temperature_degree_celsius`. The calculation is based on the
[switchgear-thermal-model](https://github.com/alliander-opensource/switchgear-thermal-model) library,
which implements the first order thermal model of IEC 62271-306. It is an optional dependency:

```bash
pip install pandapower[tdpf,plotting]
```

The switch thermal model currently requires Python 3.12 or newer. The `plotting` extra is only needed
for the plots in this notebook, which use the pandas plotting API backed by matplotlib.

If the package is not installed, the line and transformer TDPF still work unchanged; pandapower only
logs a warning and does not report switch temperatures.

In [ ]:
import numpy as np
import pandas as pd

from pandapower import pandapowerNet, runpp
from pandapower.control import ConstControl
from pandapower.create import create_buses, create_ext_grid, create_load, create_switch
from pandapower.timeseries import DFData, OutputWriter, run_timeseries
from pandapower.toolbox import compute_switch_flows

## A minimal example network

A switch has no nameplate data from which a thermal model could be derived, so the specifications are
input columns of `net.switch`:

| Column | Meaning | Default |
|---|---|---|
| `measured_temperature_rise_kelvin` | temperature rise above ambient at rated current | `75` |
| `temperature_rise_exponent` | exponent that scales the rise with the current | `1.8` |
| `thermal_time_constant_minutes` | thermal time constant in minutes | `90` |
| `air_temperature_degree_celsius` | ambient temperature in °C | `35` |

The rated current is the existing `in_ka` column. A bus-bus switch with `z_ohm > 0` is modelled as a
small branch, so it has a solved current and can be calculated.

In [ ]:
def create_switch_net(z_ohm=0.001, in_ka=0.1, air_temperature_degree_celsius=35.0, p_mw=20.0):
    net = pandapowerNet(name="tdpf_switch")
    bus1, bus2 = create_buses(net, nr_buses=2, vn_kv=[110.0, 110.0])
    create_ext_grid(net, bus1)
    create_load(net, bus2, p_mw=p_mw, q_mvar=0.2 * p_mw)
    create_switch(
        net,
        bus1,
        bus2,
        et="b",
        closed=True,
        in_ka=in_ka,
        z_ohm=z_ohm,
        measured_temperature_rise_kelvin=75.0,
        temperature_rise_exponent=1.8,
        thermal_time_constant_minutes=90.0,
        air_temperature_degree_celsius=air_temperature_degree_celsius,
    )
    return net


net = create_switch_net()
runpp(net, tdpf=True)

net.res_switch[["i_ka", "loading_percent", "temperature_degree_celsius"]]

The switch carries about 105 A at a rated current of 100 A, so it heats up above the ambient
temperature. `temperature_degree_celsius` is only created when the thermal calculation actually ran.

## Steady state and thermal inertia

Whether a steady state or a transient temperature is reported depends on the `tdpf_delay_s` parameter,
the same parameter that controls the thermal inertia of the lines and transformers:

* `tdpf_delay_s=None` (the default) is the steady state temperature for the present current.
* A finite `tdpf_delay_s` is the temperature after this time has passed, starting from ambient.

The default thermal time constant is 90 minutes, so the switch needs more than an hour to get close to
its final temperature.

In [ ]:
delays_min = [5, 15, 30, 60, 120, 240, 480, None]
temperatures = []
for delay_min in delays_min:
    net = create_switch_net()
    runpp(net, tdpf=True, tdpf_delay_s=None if delay_min is None else delay_min * 60)
    temperatures.append(net.res_switch.at[0, "temperature_degree_celsius"])

pd.Series(temperatures, index=[str(d) for d in delays_min], name="switch (°C)")

## Ambient temperature and rated current

The temperature rise is added on top of the ambient temperature, so a hotter day shifts the whole
steady state upwards. The rise itself scales with the load ratio:

In [ ]:
ambient = []
for value in (0.0, 20.0, 35.0, 45.0):
    net = create_switch_net(air_temperature_degree_celsius=value)
    runpp(net, tdpf=True)
    ambient.append(net.res_switch.at[0, "temperature_degree_celsius"])

rated = []
for value in (0.08, 0.1, 0.125, 0.16):
    net = create_switch_net(in_ka=value)
    runpp(net, tdpf=True)
    rated.append(net.res_switch.at[0, "temperature_degree_celsius"])

pd.DataFrame(
    {
        "temperature at varying ambient (°C)": ambient,
        "temperature at varying rated current (°C)": rated,
    },
    index=["case 1", "case 2", "case 3", "case 4"],
)

## Which switches are calculated?

Only closed switches with a finite positive `in_ka` and a finite solved current get a temperature. In
particular:

* a fused bus-bus switch (`z_ohm=0`) has no branch and therefore no current,
* a bus-trafo3w switch (`et="t3"`) is not supported,
* an open switch carries no current.

These switches stay `NaN`. They are skipped with a single aggregated log message, and
`compute_switch_flows` is **not** called implicitly. If the current of a fused switch is needed, call
it explicitly after the load flow:

In [ ]:
net = create_switch_net(z_ohm=0.0)
runpp(net, tdpf=True)
print(net.res_switch[["i_ka", "temperature_degree_celsius"]])

compute_switch_flows(net)
from pandapower.pf.switchgear_thermal import calculate_switch_temperatures

calculate_switch_temperatures(net)
net.res_switch[["i_ka", "temperature_degree_celsius"]]

## Time series: warming up and cooling down

In a single power flow, a finite `tdpf_delay_s` always starts from ambient. In a time series simulation,
pandapower chains the switch temperature across the time steps: the result of the previous time step is
the initial condition of the next one. This is what makes the thermal inertia physically meaningful.

In the following example the load ramps up linearly while the time step is 10 minutes.

In [ ]:
net = create_switch_net()
n_steps = 48
load_profile = pd.DataFrame(
    {
        "p_mw": np.linspace(5.0, 25.0, n_steps),
        "q_mvar": np.linspace(1.0, 5.0, n_steps),
    }
)
ds = DFData(load_profile)
ConstControl(net, "load", "p_mw", net.load.index, "p_mw", ds, recycle=False)
ConstControl(net, "load", "q_mvar", net.load.index, "q_mvar", ds, recycle=False)

ow = OutputWriter(net, output_path=None, log_variables=[])
ow.log_variable("res_switch", "temperature_degree_celsius")

run_timeseries(net, time_steps=range(n_steps), tdpf=True, tdpf_delay_s=10 * 60)

In [ ]:
temperature = ow.output["res_switch.temperature_degree_celsius"][0]

ax = temperature.plot(figsize=(6, 3.5))
ax.set_xlabel("Time step (10 min)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Switch warming up with a ramping load");

The temperature lags behind the instantaneous steady state value. That gap is the thermal inertia of the
switch, controlled by `thermal_time_constant_minutes`.

The temperature is reported, not fed back into the power flow: the switch resistance is never
modified, so voltages, flows and losses are the same as with `tdpf=False`. Only the additional result
column differs.